In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import io
from PIL import Image, ImageDraw, ImageFont

# PARAMETERS
input_shape = (64, 64, 3)
num_classes = 10
batch_size = 64
epochs_frozen = 10
epochs_unfrozen = 30  # total 40 epochs
total_epochs = epochs_frozen + epochs_unfrozen

# Data generators with augmentation and rescaling
datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
datagen_val = ImageDataGenerator(rescale=1./255)

train_path = r'E:\cnndatasets\numbers_dataset\train'
val_path = r'E:\cnndatasets\numbers_dataset\val'

train_generator = datagen_train.flow_from_directory(
    train_path,
    target_size=input_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = datagen_val.flow_from_directory(
    val_path,
    target_size=input_shape[:2],
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Custom Learning Rate Schedule with get_config
class WarmUpCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, decay_steps, alpha, warmup_steps):
        super().__init__()
        self.initial_lr = initial_lr
        self.decay_steps = decay_steps
        self.alpha = alpha
        self.warmup_steps = warmup_steps
        self.cosine_decay = tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=initial_lr,
            decay_steps=decay_steps - warmup_steps,
            alpha=alpha
        )
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        def warmup():
            return self.initial_lr * (step / warmup_steps)
        def decay():
            return self.cosine_decay(step - warmup_steps)
        return tf.cond(step < warmup_steps, warmup, decay)
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "decay_steps": self.decay_steps,
            "alpha": self.alpha,
            "warmup_steps": self.warmup_steps
        }

steps_per_epoch = train_generator.samples // batch_size
total_steps = total_epochs * steps_per_epoch
warmup_steps = int(0.1 * total_steps)

lr_schedule = WarmUpCosineDecay(
    initial_lr=1e-4,
    decay_steps=total_steps,
    alpha=0.1,
    warmup_steps=warmup_steps
)

# Build ResNet50 base model
base_model = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape,
    pooling='avg'
)

# Freeze all layers initially
for layer in base_model.layers:
    layer.trainable = False

inputs = layers.Input(shape=input_shape)
x = layers.Lambda(lambda x: x * 255.0)(inputs)  # scale 0-1 inputs to 0-255
x = base_model(x, training=False)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model_resnet = models.Model(inputs=inputs, outputs=outputs, name='resnet50_custom')

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# Compile with new optimizer instance
optimizer1 = optimizers.Adam(learning_rate=lr_schedule)
model_resnet.compile(optimizer=optimizer1, loss=loss_fn, metrics=['accuracy'])

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=7, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint('best_resnet_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

model_resnet.summary()
plot_model(
    model_resnet,
    to_file='ResNet_model_structure.png',
    show_shapes=True,
    show_layer_names=True,
    dpi=96
)

# PHASE 1: Train frozen base model
history_frozen = model_resnet.fit(
    train_generator,
    epochs=epochs_frozen,
    validation_data=val_generator,
    callbacks=callbacks_list,
    verbose=1
)

# PHASE 2: Unfreeze last 50 layers and recompile with new optimizer instance
print("Unfreezing last 50 layers of base model and continuing training.")
for layer in base_model.layers[-50:]:
    layer.trainable = True

# New optimizer instance for unfrozen training
optimizer2 = optimizers.Adam(learning_rate=lr_schedule)
model_resnet.compile(optimizer=optimizer2, loss=loss_fn, metrics=['accuracy'])

history_unfrozen = model_resnet.fit(
    train_generator,
    initial_epoch=epochs_frozen,
    epochs=total_epochs,
    validation_data=val_generator,
    callbacks=callbacks_list,
    verbose=1
)

model_resnet.save('resnet_model.h5')
model_resnet.load_weights('best_resnet_model.keras')

def save_text_as_image(text, filename, font_path=None, font_size=16, padding=10):
    lines = text.split('\n')
    font = ImageFont.load_default() if font_path is None else ImageFont.truetype(font_path, font_size)
    width = max(font.getsize(line)[0] for line in lines) + 2 * padding
    height = sum(font.getsize(line)[1] for line in lines) + 2 * padding
    img = Image.new('RGB', (width, height), color='white')
    draw = ImageDraw.Draw(img)
    y = padding
    for line in lines:
        draw.text((padding, y), line, font=font, fill='black')
        y += font.getsize(line)[1]
    img.save(filename)

# Merge histories
history = {}
for k in history_frozen.history.keys():
    history[k] = history_frozen.history[k] + history_unfrozen.history[k]

# Plot training history
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history['accuracy'], label='Train Acc')
plt.plot(history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.legend()
plt.tight_layout()
plt.savefig('ResNet_training_history.png')
plt.close()

stream = io.StringIO()
model_resnet.summary(print_fn=lambda x: stream.write(x + '\n'))
summary_str_resnet = stream.getvalue()
save_text_as_image(summary_str_resnet, 'ResNet_model_summary.png')

val_steps = val_generator.samples // batch_size
y_pred_probs_resnet = model_resnet.predict(val_generator, steps=val_steps)
y_pred_resnet = np.argmax(y_pred_probs_resnet, axis=1)
y_true_resnet = val_generator.classes[:len(y_pred_resnet)]
class_names = list(val_generator.class_indices.keys())

report_resnet = classification_report(y_true_resnet, y_pred_resnet, target_names=class_names, digits=4)
print("Classification Report:")
print(report_resnet)
save_text_as_image(report_resnet, 'ResNet_classification_report.png')

cm_resnet = confusion_matrix(y_true_resnet, y_pred_resnet)
cm_norm_resnet = cm_resnet.astype(float) / cm_resnet.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm_resnet, annot=True, fmt='.1f', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.title('Normalized Confusion Matrix (%)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('ResNet_confusion_matrix.png')
plt.close()

accuracy_resnet = accuracy_score(y_true_resnet, y_pred_resnet)
precision_resnet = precision_score(y_true_resnet, y_pred_resnet, average='weighted')
recall_resnet = recall_score(y_true_resnet, y_pred_resnet, average='weighted')
f1_resnet = f1_score(y_true_resnet, y_pred_resnet, average='weighted')
print(f"Accuracy: {accuracy_resnet:.4f}")
print(f"Precision: {precision_resnet:.4f}")
print(f"Recall: {recall_resnet:.4f}")
print(f"F1-score: {f1_resnet:.4f}")


Found 19840 images belonging to 10 classes.
Found 4960 images belonging to 10 classes.
Model: "resnet50_custom"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_24 (InputLayer)       [(None, 64, 64, 3)]       0         
                                                                 
 lambda_5 (Lambda)           (None, 64, 64, 3)         0         
                                                                 
 resnet50 (Functional)       (None, 2048)              23587712  
                                                                 
 dropout_22 (Dropout)        (None, 2048)              0         
                                                                 
 dense_29 (Dense)            (None, 256)               524544    
                                                                 
 dropout_23 (Dropout)        (None, 256)               0         
                              

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


AttributeError: 'FreeTypeFont' object has no attribute 'getsize'

In [22]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import textwrap

# Define batch size (must match training)
batch_size = 64
val_path = r'E:\cnndatasets\numbers_dataset\val'

datagen_val = ImageDataGenerator(rescale=1./255)
val_generator = datagen_val.flow_from_directory(
    val_path,
    target_size=(64, 64),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# Helper function -- renders multiline monospaced text using matplotlib for crisp output
def save_text_matplotlib(text, filename, font_size=14, line_width=120, dpi=300, pad=0.02):
    lines = []
    for l in text.split('\n'):
        lines.extend(textwrap.wrap(l, width=line_width) or [" "])
    n_lines = len(lines)
    height_per_line = font_size * 1.5 / dpi  # Height in inches per line
    fig_height = height_per_line * n_lines + pad
    fig_width = min(max(len(max(lines, key=len)) * (font_size/12) * 0.09, 8), 48)
    plt.figure(figsize=(fig_width, fig_height), dpi=dpi)
    plt.axis('off')
    plt.figtext(0, 1-pad, "\n".join(lines), wrap=True,
                fontfamily='monospace', fontsize=font_size, va="top", ha="left")
    plt.savefig(filename, bbox_inches='tight', dpi=dpi, pad_inches=pad, transparent=False)
    plt.close()

# Print model summary and save as image
stream = io.StringIO()
model_resnet.summary(print_fn=lambda x: stream.write(x + '\n'))
summary_str = stream.getvalue()
save_text_matplotlib(summary_str, 'ResNet_model_structure.png', font_size=13, line_width=130)

# Generate predictions and calculate metrics
val_steps = val_generator.samples // batch_size
y_pred_probs = model_resnet.predict(val_generator, steps=val_steps)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes[:len(y_pred)]
class_names = list(val_generator.class_indices.keys())

# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("Classification Report:\n", report)
save_text_matplotlib(report, 'ResNet_classification_report.png', font_size=18, line_width=100)

# Confusion matrix plot and save as image
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.1f',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues'
)
plt.title('Normalized Confusion Matrix (%)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('ResNet_confusion_matrix.png')
plt.close()


Found 4960 images belonging to 10 classes.
77/77 [==============================] - 87s 1s/step
Classification Report:
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       496
           1     1.0000    1.0000    1.0000       496
           2     1.0000    1.0000    1.0000       496
           3     1.0000    1.0000    1.0000       496
           4     1.0000    1.0000    1.0000       496
           5     1.0000    1.0000    1.0000       496
           6     1.0000    1.0000    1.0000       496
           7     1.0000    1.0000    1.0000       496
           8     1.0000    1.0000    1.0000       496
           9     1.0000    1.0000    1.0000       464

    accuracy                         1.0000      4928
   macro avg     1.0000    1.0000    1.0000      4928
weighted avg     1.0000    1.0000    1.0000      4928

